# **Notebook 1: Data Understanding & Exploratory Data Analysis**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED for this notebook:**
- [ ] `corporate_policies/` folder with `.md` SOP files uploaded to Colab:
  - `refund_policy.md`, `shipping_delays.md`, `password_reset.md`, `technical_troubleshooting.md`, `escalation_matrix.md`
- [ ] Internet access (to download Bitext dataset from HuggingFace)

**Files this notebook will CREATE:**
- [ ] `sampled_data.csv` — Cleaned, deduplicated, 4000-row sample of the Bitext dataset
  _(Required by: Notebook 2 — Data Preparation)_

> **Note:** Tasks 1.1 (Define Project Scope) and 1.4 (Design Methodology) are documented in your **Project Proposal PDF**, not this notebook. This notebook covers Tasks **1.2** and **1.3**.

---

## **Stage 1: Data Understanding & EDA**
### **Task 1.2: Understand and Assess Data**

#### **1.2.1 Inspect Dataset Structure [2 marks]**
**The Task:** Identify the presence of the corporate policy files within the local environment and successfully download the `bitext` customer support dataset from Hugging Face. Print the dataset information.

**Hints & Tips:**
* Use `glob` to find `.md` files in your workspace, and `load_dataset` for the bitext data.
* The dataset ID is `bitext/Bitext-customer-support-llm-chatbot-training-dataset` — it contains ~26,872 rows.
* Key columns: `instruction` (user query), `intent` (target class), `category` (broader group).

**Why we are doing it:** To ensure the local environment is properly configured with our knowledge base before reading data.
**How we are doing it:** Check the `corporate_policies` directory, load the dataset, and use `.info()`.

**Learner Inference:** If the files load correctly, your environment is stable. Seeing the 'intent' column confirms the target variable you will train your model to extract in Stage 4.

In [ ]:
# ---- Setup (run once) ----
# !pip install -q datasets scikit-learn seaborn matplotlib pandas
import os, glob, json, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from datasets import load_dataset

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # keep identical in ALL notebooks

# --- Locate SOP files (unzip Dataset.zip first if the folder isn't there yet) ---
if not glob.glob("**/sop_documents/*.md", recursive=True):
    for z in ["Dataset.zip", "Dataset_1.zip"]:
        if os.path.exists(z):
            zipfile.ZipFile(z).extractall(".")
            break

sop_files = sorted(glob.glob("**/sop_documents/*.md", recursive=True))
assert sop_files, "No SOP .md files found - upload/unzip Dataset.zip into the Colab workspace."
SOP_DIR = os.path.dirname(sop_files[0])
print(f"SOP directory: {SOP_DIR}  |  {len(sop_files)} files")
for f in sop_files:
    print("  -", os.path.basename(f))

required = {"refund_policy.md", "shipping_delays.md", "password_reset.md",
            "technical_troubleshooting.md", "escalation_matrix.md"}
missing = required - {os.path.basename(f) for f in sop_files}
print("Required SOPs missing:", missing or "none")

# --- Load Bitext ---
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df_full = ds["train"].to_pandas()
print("\nDataset info:")
df_full.info()
display(df_full[["instruction", "intent", "category"]].head())

#### **1.2.2 Quantify Dataset Characteristics [3 marks]**
**The Task:** Calculate the original size of the dataset, identify the number of unique intents, and randomly sample the dataset down to 4,000 rows.

**Hints & Tips:**
* Use `len()`, `.nunique()`, and `.sample(4000, random_state=42)`.
* `random_state=42` ensures reproducibility — everyone gets the same sample.
* With 4000 rows and 27 intents, you'll get ~148 examples per intent on average.

**Parameter Tuning:**
* `sample(n)` — try 1000–5000 per the workflow spec:
  - `1000`: Fast (~1 min), lower accuracy
  - `4000`: Moderate (~3 min), better accuracy (recommended)
  - `5000`: Slower, best accuracy, may hit T4 memory limits

**Learner Inference:** Capping at 4000 rows keeps your Stage 4 training loop to minutes while providing enough examples per intent.

In [ ]:
# 1.2.2 - Dataset size, unique intents, sample down to 4,000
print(f"Original rows        : {len(df_full):,}")
print(f"Unique intents       : {df_full['intent'].nunique()}")
print(f"Unique categories    : {df_full['category'].nunique()}")

df = df_full.sample(4000, random_state=SEED).reset_index(drop=True)
print(f"Sampled rows         : {len(df):,}  (avg ~{len(df)/df['intent'].nunique():.0f} per intent)")

# Prompt-length characteristics (word counts; token counts follow in NB2)
df["n_words"] = df["instruction"].str.split().str.len()
print("\nPrompt length (words):")
print(df["n_words"].describe().round(1))

#### **1.2.3 Validate Sample Quality [2 marks]**
**The Task:** Identify and remove records containing null values or duplicated customer instructions using normalised deduplication.

**Hints & Tips:**
* Use `.isnull().sum()` to check nulls.
* Simple `drop_duplicates` misses near-dupes like `"Where is my order?"` vs `"where is my order?"`.
* Normalise to lowercase + strip whitespace BEFORE deduplicating to catch these.
* The Bitext dataset typically has ~15 near-duplicates in a 4000-row sample.

**Learner Inference:** Cleaning now prevents "garbage-in, garbage-out". Duplicate prompts can cause leakage between train and test sets later.

In [ ]:
# 1.2.3 - Nulls, empty strings, normalised near-duplicates
print("Null counts:")
print(df[["instruction", "intent", "category"]].isnull().sum())

before = len(df)
df = df.dropna(subset=["instruction", "intent", "category"])
df = df[df["instruction"].astype(str).str.strip() != ""]
print(f"\nRemoved {before - len(df)} null/empty rows")

# Normalised dedup: lowercase + strip BEFORE comparing
before = len(df)
df["_norm"] = df["instruction"].str.lower().str.strip()
exact_dupes = df.duplicated(subset=["instruction"]).sum()
norm_dupes  = df.duplicated(subset=["_norm"]).sum()
print(f"Exact duplicates      : {exact_dupes}")
print(f"Normalised duplicates : {norm_dupes}  (extra caught by normalising: {norm_dupes - exact_dupes})")
df = df.drop_duplicates(subset=["_norm"]).drop(columns="_norm").reset_index(drop=True)
print(f"Removed {before - len(df)} near-duplicates -> {len(df):,} rows remain")

# Prompt-solution alignment: each intent should map to exactly one category
align = df.groupby("intent")["category"].nunique()
print("\nIntents mapped to >1 category:", int((align > 1).sum()))
assert 1000 <= len(df) <= 5000, "Row count outside the 1,000-5,000 requirement"

### **Task 1.3: Perform Exploratory Data Analysis**

#### **1.3.1 Perform Univariate Analysis [2 marks]**
**The Task:** Plot the top 10 most frequent customer intents using a horizontal bar chart.

**Hints & Tips:**
* Use a Seaborn `countplot` with `order=df['intent'].value_counts().index[:10]`.
* Check if any single intent dominates >80% of data — signals severe class imbalance.
* `palette="viridis"` works well for accessibility.

**Learner Inference:** This visual reveals class imbalance. If one intent dominates, your fine-tuned router becomes biased toward guessing it.

In [ ]:
# 1.3.1 - Top 10 intents (horizontal bar)
top10 = df["intent"].value_counts().index[:10]
plt.figure(figsize=(9, 5))
sns.countplot(data=df, y="intent", order=top10, palette="viridis")
plt.title("Top 10 Customer Intents")
plt.xlabel("Count"); plt.ylabel("Intent")
plt.tight_layout(); plt.show()

share = df["intent"].value_counts(normalize=True)
print(f"Most frequent intent : {share.index[0]} ({share.iloc[0]:.1%})")
print(f"Min / max per intent : {df['intent'].value_counts().min()} / {df['intent'].value_counts().max()}")
print("Severe imbalance (>80% one class):", share.iloc[0] > 0.80)

#### **1.3.2 Perform Bivariate Analysis [2 marks]**
**The Task:** Read the text from the uploaded Markdown SOP files and fit a `TfidfVectorizer` (with `stop_words='english'`) to transform them into vectors.

**Hints & Tips:**
* Read files into a list of strings first using `open(f, 'r').read()`.
* `TfidfVectorizer(stop_words="english")` removes common words, keeping meaningful terms.
* The resulting matrix shape should be `(num_docs, num_unique_terms)`.

**Learner Inference:** This transforms raw text into math, letting you prove whether your policies are distinct enough for an AI to tell apart.

In [ ]:
# 1.3.2 - TF-IDF on SOP documents
from sklearn.feature_extraction.text import TfidfVectorizer

sop_names, sop_texts = [], []
for f in sop_files:
    with open(f, "r", encoding="utf-8") as fh:
        sop_texts.append(fh.read())
    sop_names.append(os.path.basename(f).replace(".md", ""))

tfidf = TfidfVectorizer(stop_words="english")
sop_matrix = tfidf.fit_transform(sop_texts)
print("TF-IDF matrix shape (docs, terms):", sop_matrix.shape)
print("Sample terms:", list(tfidf.get_feature_names_out()[:15]))

#### **1.3.3 Generate Visualisations [2 marks]**
**The Task:** Calculate cosine similarity between the TF-IDF vectors of the SOP documents and plot an annotated heatmap.

**Hints & Tips:**
* Use `cosine_similarity` from sklearn and `sns.heatmap` with `annot=True`.
* Diagonal should always be 1.0 (document vs itself).
* Off-diagonal >0.5 = high overlap (RAG may struggle); <0.3 = low overlap (good retrieval).

**Learner Inference:** High overlap (dark off-diagonal squares) means the RAG system struggles to fetch the right doc. Low overlap proves retrieval will be accurate.

In [ ]:
# 1.3.3 - Cosine-similarity heatmap
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(sop_matrix)
plt.figure(figsize=(10, 8))
sns.heatmap(sim, annot=True, fmt=".2f", cmap="viridis",
            xticklabels=sop_names, yticklabels=sop_names)
plt.title("SOP Document Cosine Similarity (TF-IDF)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

# Rank most-overlapping pairs (retrieval-confusion risk)
pairs = [(sop_names[i], sop_names[j], sim[i, j])
         for i in range(len(sop_names)) for j in range(i + 1, len(sop_names))]
pairs.sort(key=lambda p: -p[2])
print("Top 5 most similar SOP pairs:")
for a, b, s in pairs[:5]:
    print(f"  {a:28s} <-> {b:28s} {s:.3f}")
print("\nPairs with similarity > 0.5:", sum(s > 0.5 for _, _, s in pairs))

---
## Save Artifacts for Notebook 2

**IMPORTANT:** This cell saves the cleaned DataFrame. Notebook 2 depends on this file.

In [ ]:
# Save artifacts for Notebook 2
df_out = df[["instruction", "intent", "category"]].copy()
df_out.to_csv("sampled_data.csv", index=False)

config = {"MODEL_ID": MODEL_ID, "SEED": SEED, "SOP_DIR": SOP_DIR,
          "EMBED_MODEL": "sentence-transformers/all-MiniLM-L6-v2",
          "n_rows": len(df_out), "n_intents": int(df_out["intent"].nunique())}
json.dump(config, open("project_config.json", "w"), indent=2)
print("Saved sampled_data.csv:", df_out.shape, "| project_config.json")

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify all items before proceeding to Notebook 2.**

- [ ] Bitext dataset loaded and `.info()` printed
- [ ] Dataset sampled to 4000 rows (or chosen size 1000–5000)
- [ ] Nulls inspected, near-duplicates removed via normalised dedup
- [ ] Top 10 intent bar chart rendered
- [ ] TF-IDF vectorisation completed on SOP documents
- [ ] Cosine similarity heatmap rendered
- [ ] **`sampled_data.csv` saved to disk** ← _CRITICAL for Notebook 2_
- [ ] `corporate_policies/` folder still accessible ← _Needed in Notebooks 2 and 4_

**If any item is unchecked, fix it before moving on.**